# Execução dos experimentos (Google Colab)

Este notebook executa a comparação entre as ferramentas AutoML (AutoGluon, FLAML, H2O AutoML e TPOT) em conjuntos de dados de regressão, usando o código de `common/runner.py`.

**Protocolo:** partição 80/20 com `random_state = semente`; sementes 42 a 46; orçamento de busca de 300 s por execução; métricas RMSE, MAE e R². Cada execução gera um arquivo `{ferramenta}__{dataset}__seed{n}.json` (e as previsões em `_pred.npz`) na pasta de resultados.

**Como usar**
1. Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4. A GPU não é obrigatória; o estudo usou a T4 e, entre as ferramentas, apenas o H2O a utilizou.
2. Ajuste os parâmetros na célula de configuração (conjuntos de dados, ferramentas, sementes).
3. Execute as células em ordem.
4. A coleta é **retomável**: o runner pula toda execução que já tem arquivo `.json` (concluída) ou `.falha` (falhou). Se a sessão do Colab cair, abra o notebook de novo e rode a partir da célula de instalação.
5. Os resultados são gravados no Google Drive, em `MyDrive/TCC_resultados`.

**Tempo estimado:** cada execução leva cerca de 5 a 6 minutos. Os 4 conjuntos da etapa de ampliação (4 ferramentas × 4 datasets × 5 sementes = 80 execuções) levam cerca de 7 horas, e a versão gratuita do Colab interrompe sessões longas, então a coleta costuma exigir várias sessões.

**Sobre falhas:** cada execução roda em um processo separado, com um limite de segurança de 1.200 s. Esse limite não é o orçamento de busca (300 s); serve apenas para encerrar execuções travadas. Quando uma execução falha, o motivo fica registrado em um arquivo `.falha`, e ela não é repetida nas retomadas. Para tentar de novo, apague o arquivo `.falha` ou use `RETENTAR_FALHAS = True`.

## 1. Configuração

In [ ]:
# Conjuntos de dados da coleta (nomes conforme common/datasets_meta.py)
DATASETS_INICIAIS = ["concrete", "abalone", "mercedes", "wine_quality",
                     "brazilian_houses", "bike_sharing", "california", "diamonds"]
DATASETS_AMPLIACAO = ["kin8nm", "news_popularity", "santander", "black_friday"]

DATASETS = DATASETS_AMPLIACAO   # ou DATASETS_INICIAIS, ou a soma das duas listas
TOOLS = ["flaml", "autogluon", "h2o", "tpot"]
SEEDS = range(42, 47)

BUDGET = 300           # segundos de busca por execução, igual para todas as ferramentas
TIMEOUT_S = 1200       # limite de segurança por execução (encerra processos travados)
RETENTAR_FALHAS = False

DIR = "/content/drive/MyDrive/TCC_resultados"   # pasta de resultados no Drive

## 2. Instalação das versões fixas

As versões são as mesmas usadas em todo o estudo; não misture versões entre execuções. O Python do Colab deve ser o 3.13.

In [ ]:
!pip install -q --default-timeout=200 autogluon.tabular==1.5.0 catboost==1.2.10 flaml==2.7.0 h2o==3.46.0.12 tpot==1.1.0
# Se o Colab pedir, reinicie a sessão (Ambiente de execução > Reiniciar sessão) e rode as células seguintes.

## 3. Código do projeto

O código (runner, carga dos datasets e registro do ambiente) é obtido do repositório do projeto.

In [ ]:
import os, sys, subprocess

REPO = "/content/tcc-automl-regressao"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/PabloFernan/tcc-automl-regressao", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
sys.path.append(f"{REPO}/common")
print("Código em:", REPO)

## 4. Google Drive e registro do ambiente

Cada sessão grava um arquivo `ambiente_AAAAMMDD.json` com as versões, o Python, a CPU e a GPU. Confira se `gpu_disponivel` é `True` quando a intenção é reproduzir as condições do estudo (T4).

In [ ]:
from datetime import date
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DIR, exist_ok=True)

from diagnostico import registrar
amb = registrar(f"{DIR}/ambiente_{date.today():%Y%m%d}.json")
print("GPU disponível:", amb.get("gpu_disponivel"))

## 5. Verificação dos conjuntos de dados (opcional, cerca de 2 min)

Baixa os conjuntos escolhidos e confere o número de linhas e atributos. Valores esperados para os 4 conjuntos da etapa de ampliação: kin8nm 8192×8, news_popularity 39644×59, santander 4459×4991 e black_friday 166821×9.

In [ ]:
!cd {REPO} && python common/datasets_meta.py {" ".join(DATASETS)}

## 6. Progresso atual

In [ ]:
import glob, collections

por_seed = collections.Counter()
for d in DATASETS:
    for f in glob.glob(f"{DIR}/*__{d}__seed*.json"):
        por_seed[f.split("seed")[1].split(".")[0]] += 1

esperado = len(TOOLS) * len(DATASETS)   # execuções por semente
for s in SEEDS:
    print(f"semente {s}: {por_seed.get(str(s), 0)}/{esperado} concluídas")

falhas = [f for d in DATASETS for f in glob.glob(f"{DIR}/*__{d}__seed*.falha")]
print("Falhas registradas:", len(falhas))

## 7. Execução

O laço externo é a semente: se a sessão cair, já existe uma comparação completa com menos sementes. Rode a célula de novo para continuar de onde parou.

In [ ]:
import runner

print("Parâmetros do TPOT:", runner.TPOT_PARAMS)   # esperado: {'cv': 3, 'max_eval_time_mins': 1}
runner.executar_tudo(TOOLS, DATASETS, seeds=SEEDS, budget=BUDGET, dir_=DIR,
                     retentar_falhas=RETENTAR_FALHAS, timeout_s=TIMEOUT_S)

## 8. Resumo da coleta

Mostra o que foi concluído, o que falhou (com o motivo) e o tempo e a memória de cada execução.

In [ ]:
import json
import pandas as pd

linhas = []
for d in DATASETS:
    for f in sorted(glob.glob(f"{DIR}/*__{d}__seed*.json")):
        r = json.load(open(f, encoding="utf-8"))
        linhas.append((r["framework"], r["dataset"], r["seed"], r["tempo_real_s"],
                       round(r["ram_pico_mb"]), round(r["ram_delta_mb"]), r["rmse"]))
df = pd.DataFrame(linhas, columns=["ferramenta", "dataset", "semente", "tempo_s", "ram_pico_mb", "ram_delta_mb", "rmse"])

pd.set_option("display.width", 200)
print(df.to_string(index=False))
print("\nTempo total das execuções concluídas: %.1f h" % (df.tempo_s.sum() / 3600))
print(df.groupby(["dataset", "ferramenta"]).tempo_s.agg(["count", "mean", "max"]).round(0))

previstas = [(t, d, s) for t in TOOLS for d in DATASETS for s in SEEDS]
sem_resultado = [(t, d, s) for t, d, s in previstas
                 if not os.path.exists(f"{DIR}/{t}__{d}__seed{s}.json")]
print(f"\nConcluídas: {len(previstas) - len(sem_resultado)}/{len(previstas)}")

for f in sorted(glob.glob(f"{DIR}/*.falha")):
    falha = json.load(open(f, encoding="utf-8"))
    print("FALHA:", falha["framework"], falha["dataset"], falha["seed"], "|", falha["erro"][:150])

## Depois da coleta

1. Copie os arquivos `.json`, `_pred.npz`, `.falha` e `ambiente_*.json` da pasta `TCC_resultados` do Drive para a pasta `resultados/` do repositório.
2. Regere as tabelas, os testes estatísticos e as figuras com `python common/analise_final.py` (e `python common/analise_brazilian.py` para a análise do Brazilian Houses).

**Para dar continuidade ao estudo:**
- Para incluir um novo conjunto de dados, adicione-o ao catálogo `CANDIDATOS` em `common/datasets_meta.py` (fonte, identificador no OpenML e nome) e coloque o nome na lista `DATASETS` acima.
- Para incluir uma nova ferramenta, implemente uma função com a mesma interface das existentes em `common/runner.py` e registre-a no dicionário `FERRAMENTAS`.
- Execuções feitas em um mesmo processo (como as dos 8 primeiros conjuntos do estudo) têm o aumento de RAM influenciado pela ordem de execução; para medir memória, use o runner atual, que isola cada execução em um processo próprio.
- A lista de execuções sem isolamento do estudo está em `resultados/rodada2_execucoes_nao_isoladas.txt`.